# 1. Purpose, scope, and locked-test warning

This notebook freezes the completed ten-seed MentalRoBERTa Phase 3 experiment and performs the predeclared Phase 4 residual-boundary analysis from frozen validation predictions.

It runs entirely on CPU. It does not load model checkpoints, perform inference or training, read raw social-media text, or access the locked test. Phase 4 can identify provisional criterion-1 candidates, but final hard pairs remain unfrozen until the separate Phase 5 bootstrap criterion is evaluated.


# 2. User configuration

Place this notebook beside the `mental_roberta_10_seeds results` directory, or change the paths below. No other cell should require editing.


In [ ]:
# USER CONFIGURATION
from pathlib import Path

INPUT_DIR = Path("mental_roberta_10_seeds results")
OUTPUT_DIR = Path("phase4_boundary_analysis_results")
OPTIONAL_SUMMARY_PATH = INPUT_DIR / "mentalroberta_full_10seed_summary.txt"

RESULT_ZIP_PATH = Path("mentalroberta_phase3_freeze_phase4_boundary_analysis_RESULTS.zip")
SOURCE_NOTEBOOK_PATH = Path("step-4-mentalroberta-10seeds.ipynb")
MASTER_PLAN_PATH = Path("MentalDistress_Master_Research_Plan_PREEXPERIMENT_REVIEWER_RESILIENT_2026-08-22.md")


# 3. Imports, deterministic settings, and version capture


In [ ]:
import json
import os
import platform
import sys
import tempfile

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(0)

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "nbformat": nbformat.__version__,
    "execution_device": "CPU",
    "network_required": False,
}
print(json.dumps(environment, indent=2, sort_keys=True))


{
  "execution_device": "CPU",
  "nbformat": "5.10.4",
  "network_required": false,
  "numpy": "2.5.2",
  "pandas": "3.0.3",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "python": "3.14.4"
}


# 4. Pure helper functions

The complete validated analysis implementation is embedded below. The notebook has no dependency on project-local Python modules.


In [ ]:
"""Deterministic Phase 3 freeze and Phase 4 boundary-analysis helpers."""

from __future__ import annotations

import csv
import hashlib
import io
import json
from itertools import combinations
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd


LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
EXPECTED_SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
EXPECTED_MODEL_REVISION = "610e9d35d8f2550d713579077242d5e523209cea"
REQUIRED_ARCHIVE_MEMBERS = {
    "artifacts/config.json",
    "artifacts/dataset_integrity.json",
    "artifacts/validation_predictions.csv",
    "artifacts/validation_metrics.json",
    "artifacts/sha256_manifest.csv",
}
METRIC_KEYS = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "mcc",
]
TRAINING_CONFIG_KEYS = [
    "run_mode",
    "max_length",
    "train_batch_size",
    "eval_batch_size",
    "grad_accum_steps",
    "epochs",
    "patience",
    "learning_rate",
    "weight_decay",
    "warmup_ratio",
    "max_grad_norm",
]


def _validate_confusion(confusion, labels):
    matrix = np.asarray(confusion)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("confusion matrix must be square")
    if matrix.shape[0] != len(labels):
        raise ValueError("label count must match confusion matrix dimensions")
    if not np.issubdtype(matrix.dtype, np.number) or not np.isfinite(matrix).all():
        raise ValueError("confusion matrix must contain finite numbers")
    if (matrix < 0).any() or not np.equal(matrix, np.floor(matrix)).all():
        raise ValueError("confusion matrix must contain non-negative integers")
    matrix = matrix.astype(np.int64, copy=False)
    supports = matrix.sum(axis=1)
    if (supports <= 0).any():
        raise ValueError("each class must have positive support")
    if len(labels) != len(set(labels)):
        raise ValueError("labels must be unique")
    return matrix, supports


def compute_pair_metrics(confusion, labels, seed):
    """Compute BCR and error allocation for every unordered label pair."""
    matrix, supports = _validate_confusion(confusion, labels)
    total_errors = int(matrix.sum() - np.trace(matrix))
    rows = []
    for i, j in combinations(range(len(labels)), 2):
        confusion_i_to_j = int(matrix[i, j])
        confusion_j_to_i = int(matrix[j, i])
        pair_errors = confusion_i_to_j + confusion_j_to_i
        bcr = 0.5 * (
            confusion_i_to_j / int(supports[i])
            + confusion_j_to_i / int(supports[j])
        )
        rows.append(
            {
                "seed": int(seed),
                "pair": f"{labels[i]}|{labels[j]}",
                "class_i": labels[i],
                "class_j": labels[j],
                "support_i": int(supports[i]),
                "support_j": int(supports[j]),
                "confusion_i_to_j": confusion_i_to_j,
                "confusion_j_to_i": confusion_j_to_i,
                "pair_errors": pair_errors,
                "bcr": float(bcr),
                "total_errors": total_errors,
                "pair_error_share": float(pair_errors / total_errors) if total_errors else 0.0,
            }
        )
    return pd.DataFrame(rows)


def rank_pairs(pair_metrics):
    """Assign descending average BCR ranks independently within each seed."""
    required = {"seed", "pair", "bcr"}
    missing = required.difference(pair_metrics.columns)
    if missing:
        raise ValueError(f"pair metrics missing columns: {sorted(missing)}")
    frame = pair_metrics.copy()
    if not np.isfinite(frame["bcr"].to_numpy(dtype=float)).all():
        raise ValueError("BCR values must be finite")
    frame["rank"] = frame.groupby("seed", sort=False)["bcr"].rank(
        method="average", ascending=False
    )
    return frame.sort_values(["seed", "rank", "pair"], kind="mergesort").reset_index(drop=True)


def summarize_pairs(per_seed_pairs):
    """Summarize BCRs, ranks, and error allocation across seeds."""
    required = {"pair", "bcr", "rank", "pair_errors", "pair_error_share"}
    missing = required.difference(per_seed_pairs.columns)
    if missing:
        raise ValueError(f"ranked pair metrics missing columns: {sorted(missing)}")

    rows = []
    for pair, group in per_seed_pairs.groupby("pair", sort=True):
        bcr = group["bcr"].to_numpy(dtype=float)
        ranks = group["rank"].to_numpy(dtype=float)
        rows.append(
            {
                "pair": pair,
                "seed_count": int(len(group)),
                "mean_bcr": float(np.mean(bcr)),
                "sample_sd_bcr": float(np.std(bcr, ddof=1)) if len(bcr) > 1 else 0.0,
                "median_bcr": float(np.median(bcr)),
                "bcr_q1": float(np.quantile(bcr, 0.25)),
                "bcr_q3": float(np.quantile(bcr, 0.75)),
                "mean_rank": float(np.mean(ranks)),
                "median_rank": float(np.median(ranks)),
                "top1_count": int(np.sum(ranks <= 1.0)),
                "top1_frequency": float(np.mean(ranks <= 1.0)),
                "top3_count": int(np.sum(ranks <= 3.0)),
                "top3_frequency": float(np.mean(ranks <= 3.0)),
                "mean_pair_errors": float(group["pair_errors"].mean()),
                "mean_pair_error_share": float(group["pair_error_share"].mean()),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["median_rank", "mean_bcr", "pair"], ascending=[True, False, True], kind="mergesort"
    ).reset_index(drop=True)


def _validate_rank_matrix(rank_matrix):
    frame = pd.DataFrame(rank_matrix).copy()
    values = frame.to_numpy(dtype=float)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 2:
        raise ValueError("rank matrix must contain at least two pairs and two seeds")
    if not np.isfinite(values).all():
        raise ValueError("rank matrix must contain only finite values")
    return frame, values


def kendalls_w(rank_matrix):
    """Calculate Kendall's W for pairs (rows) ranked by seeds (columns)."""
    frame, values = _validate_rank_matrix(rank_matrix)
    pair_count, seed_count = values.shape
    expected_rank_sum = pair_count * (pair_count + 1) / 2.0
    column_sums = values.sum(axis=0)
    if not np.allclose(column_sums, expected_rank_sum, rtol=0.0, atol=1e-9):
        raise ValueError("each seed column must contain complete ranks")

    rank_sums = values.sum(axis=1)
    centered_sum_squares = float(
        np.sum((rank_sums - seed_count * (pair_count + 1) / 2.0) ** 2)
    )
    tie_correction = 0.0
    for column in values.T:
        _, counts = np.unique(column, return_counts=True)
        tie_correction += float(np.sum(counts**3 - counts))

    denominator = (
        seed_count**2 * (pair_count**3 - pair_count)
        - seed_count * tie_correction
    )
    if denominator <= 0:
        raise ValueError("Kendall W denominator is not positive")
    coefficient = 12.0 * centered_sum_squares / denominator
    coefficient = float(np.clip(coefficient, 0.0, 1.0))
    return {
        "method": "Kendall coefficient of concordance with tie correction",
        "kendalls_w": coefficient,
        "pair_count": int(pair_count),
        "seed_count": int(seed_count),
        "tie_correction": float(tie_correction),
    }


def pairwise_seed_spearman(rank_matrix):
    """Calculate every pairwise Spearman correlation between seed rankings."""
    frame, _ = _validate_rank_matrix(rank_matrix)
    rows = []
    for seed_a, seed_b in combinations(frame.columns.tolist(), 2):
        a = frame[seed_a].to_numpy(dtype=float)
        b = frame[seed_b].to_numpy(dtype=float)
        rho = float(np.corrcoef(a, b)[0, 1])
        if not np.isfinite(rho):
            raise ValueError("Spearman correlation is not finite")
        rows.append({"seed_a": seed_a, "seed_b": seed_b, "spearman_rho": rho})
    comparisons = pd.DataFrame(rows)
    values = comparisons["spearman_rho"].to_numpy(dtype=float)
    summary = {
        "method": "Pearson correlation applied to complete average-rank vectors",
        "comparison_count": int(len(comparisons)),
        "median_spearman_rho": float(np.median(values)),
        "spearman_rho_q1": float(np.quantile(values, 0.25)),
        "spearman_rho_q3": float(np.quantile(values, 0.75)),
        "minimum_spearman_rho": float(np.min(values)),
        "maximum_spearman_rho": float(np.max(values)),
    }
    return comparisons, summary


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def discover_archives(input_dir):
    input_path = Path(input_dir)
    if not input_path.is_dir():
        raise ValueError(f"input directory does not exist: {input_path}")
    archives = sorted(input_path.glob("mentalroberta_full_seed*_len512_*_ARTIFACTS.zip"))
    if len(archives) != 10:
        raise ValueError(f"expected exactly 10 artifact archives, found {len(archives)}")
    return archives


def validate_archive(path):
    archive_path = Path(path)
    if not archive_path.is_file():
        raise ValueError(f"archive does not exist: {archive_path}")
    try:
        with zipfile.ZipFile(archive_path) as archive:
            corrupt_member = archive.testzip()
            if corrupt_member is not None:
                raise ValueError(f"ZIP integrity failure in {corrupt_member}")
            member_names = set(archive.namelist())
            missing = REQUIRED_ARCHIVE_MEMBERS.difference(member_names)
            if missing:
                raise ValueError(f"archive missing required members: {sorted(missing)}")
            manifest_text = archive.read("artifacts/sha256_manifest.csv").decode("utf-8-sig")
            manifest_rows = list(csv.DictReader(io.StringIO(manifest_text)))
            if not manifest_rows:
                raise ValueError("artifact manifest is empty")
            validated = 0
            for row in manifest_rows:
                relative = row.get("path", "")
                member = f"artifacts/{relative}"
                if member not in member_names:
                    raise ValueError(f"manifest member is absent: {member}")
                payload = archive.read(member)
                expected_bytes = int(row["bytes"])
                if len(payload) != expected_bytes:
                    raise ValueError(f"manifest byte-count mismatch: {member}")
                actual_hash = hashlib.sha256(payload).hexdigest()
                if actual_hash != row["sha256"]:
                    raise ValueError(f"manifest SHA-256 mismatch: {member}")
                validated += 1
    except zipfile.BadZipFile as exc:
        raise ValueError(f"invalid ZIP archive: {archive_path}") from exc
    return {
        "archive_path": archive_path,
        "archive_name": archive_path.name,
        "archive_bytes": int(archive_path.stat().st_size),
        "archive_sha256": sha256_file(archive_path),
        "validated_manifest_members": int(validated),
    }


def _read_json_member(archive, name):
    try:
        return json.loads(archive.read(name).decode("utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError, KeyError) as exc:
        raise ValueError(f"invalid JSON member: {name}") from exc


def validate_predictions(predictions, labels=LABELS, expected_rows=1490):
    required = {"original_row", "true_label", "true_id", "predicted_id", "predicted_label"}
    probability_columns = [f"prob_{label}" for label in labels]
    required.update(probability_columns)
    missing = required.difference(predictions.columns)
    if missing:
        raise ValueError(f"prediction table missing columns: {sorted(missing)}")
    if len(predictions) != expected_rows:
        raise ValueError(f"prediction row count mismatch: expected {expected_rows}, found {len(predictions)}")
    if predictions["original_row"].isna().any() or not predictions["original_row"].is_unique:
        raise ValueError("original_row values must be present and unique")

    expected_label2id = {label: index for index, label in enumerate(labels)}
    true_ids = predictions["true_id"].to_numpy()
    predicted_ids = predictions["predicted_id"].to_numpy()
    if not np.equal(true_ids, np.floor(true_ids)).all() or not np.equal(predicted_ids, np.floor(predicted_ids)).all():
        raise ValueError("label identifiers must be integers")
    true_ids = true_ids.astype(int)
    predicted_ids = predicted_ids.astype(int)
    if not np.isin(true_ids, np.arange(len(labels))).all() or not np.isin(predicted_ids, np.arange(len(labels))).all():
        raise ValueError("label identifiers are outside the locked label order")
    expected_true_labels = np.array([labels[index] for index in true_ids], dtype=object)
    expected_predicted_labels = np.array([labels[index] for index in predicted_ids], dtype=object)
    if not np.array_equal(predictions["true_label"].astype(str).to_numpy(), expected_true_labels):
        raise ValueError("true labels do not match true identifiers")
    if not np.array_equal(predictions["predicted_label"].astype(str).to_numpy(), expected_predicted_labels):
        raise ValueError("predicted labels do not match predicted identifiers")
    if set(predictions["true_label"].unique()) != set(labels):
        raise ValueError("prediction table does not contain the complete locked label set")

    probabilities = predictions[probability_columns].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all():
        raise ValueError("probabilities must be finite")
    if (probabilities < 0).any() or (probabilities > 1).any():
        raise ValueError("probabilities must lie in [0, 1]")
    if not np.allclose(probabilities.sum(axis=1), 1.0, rtol=0.0, atol=1e-5):
        raise ValueError("probabilities must sum to one")
    if not np.array_equal(probabilities.argmax(axis=1), predicted_ids):
        raise ValueError("probability argmax does not match predicted_id")
    return {
        "row_count": int(len(predictions)),
        "original_rows_unique": True,
        "probabilities_valid": True,
        "label_order": list(labels),
        "label2id": expected_label2id,
    }


def _confusion_from_ids(true_ids, predicted_ids, class_count):
    matrix = np.zeros((class_count, class_count), dtype=np.int64)
    np.add.at(matrix, (true_ids, predicted_ids), 1)
    return matrix


def recompute_metrics(predictions, labels=LABELS):
    true_ids = predictions["true_id"].to_numpy(dtype=int)
    predicted_ids = predictions["predicted_id"].to_numpy(dtype=int)
    confusion = _confusion_from_ids(true_ids, predicted_ids, len(labels))
    support = confusion.sum(axis=1).astype(float)
    predicted_support = confusion.sum(axis=0).astype(float)
    true_positive = np.diag(confusion).astype(float)
    precision = np.divide(
        true_positive,
        predicted_support,
        out=np.zeros_like(true_positive),
        where=predicted_support != 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros_like(true_positive),
        where=support != 0,
    )
    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) != 0,
    )
    sample_count = float(confusion.sum())
    correct = float(np.trace(confusion))
    numerator = correct * sample_count - float(np.dot(predicted_support, support))
    denominator = np.sqrt(
        (sample_count**2 - float(np.dot(predicted_support, predicted_support)))
        * (sample_count**2 - float(np.dot(support, support)))
    )
    mcc = numerator / denominator if denominator else 0.0
    per_class = pd.DataFrame(
        {
            "label": labels,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support.astype(int),
        }
    )
    return {
        "accuracy": correct / sample_count,
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "weighted_f1": float(np.average(f1, weights=support)),
        "mcc": float(mcc),
        "confusion_matrix": confusion,
        "per_class": per_class,
    }


def load_run(path, expected_rows=1490, allow_test_access_flag=False):
    archive_validation = validate_archive(path)
    with zipfile.ZipFile(path) as archive:
        config = _read_json_member(archive, "artifacts/config.json")
        integrity = _read_json_member(archive, "artifacts/dataset_integrity.json")
        stored_metrics = _read_json_member(archive, "artifacts/validation_metrics.json")
        predictions = pd.read_csv(io.BytesIO(archive.read("artifacts/validation_predictions.csv")))
    validate_predictions(predictions, LABELS, expected_rows=expected_rows)
    if bool(config.get("test_accessed")) and not allow_test_access_flag:
        raise ValueError("run config records test_accessed=true")
    recomputed = recompute_metrics(predictions, LABELS)
    for key in METRIC_KEYS:
        if key not in stored_metrics:
            raise ValueError(f"stored validation metrics missing {key}")
        if not np.isclose(recomputed[key], stored_metrics[key], rtol=0.0, atol=1e-12):
            raise ValueError(f"recomputed metric mismatch for {key}")
    return {
        **archive_validation,
        "seed": int(config["seed"]),
        "run_id": str(config["run_id"]),
        "config": config,
        "integrity": integrity,
        "stored_metrics": stored_metrics,
        "recomputed_metrics": recomputed,
        "predictions": predictions,
        "warnings": ["full-run repeated-fit reproducibility was not performed"],
    }


def _config_model(config):
    model = config.get("model") or config.get("resolved_model") or {}
    return {
        "model_key": model.get("model_key"),
        "source": model.get("source"),
        "revision": model.get("revision"),
    }


def validate_run_collection(
    runs,
    expected_seeds=EXPECTED_SEEDS,
    expected_rows=1490,
    expected_train_rows=None,
):
    if len(runs) != len(expected_seeds):
        raise ValueError(f"run count mismatch: expected {len(expected_seeds)}, found {len(runs)}")
    observed_seeds = [int(run["seed"]) for run in runs]
    if len(set(observed_seeds)) != len(observed_seeds) or sorted(observed_seeds) != sorted(expected_seeds):
        raise ValueError(f"observed seed set does not match expected seed set: {observed_seeds}")

    reference = sorted(runs, key=lambda item: item["seed"])[0]
    reference_config = reference["config"]
    reference_model = _config_model(reference_config)
    reference_hashes = reference_config.get("data_sha256", {})
    reference_rows = reference["predictions"][["original_row", "true_id", "true_label"]].reset_index(drop=True)
    reference_training = {key: reference_config.get(key) for key in TRAINING_CONFIG_KEYS}
    expected_label2id = {label: index for index, label in enumerate(LABELS)}
    expected_id2label = {str(index): label for index, label in enumerate(LABELS)}

    for run in runs:
        config = run["config"]
        if int(config.get("seed")) != int(run["seed"]):
            raise ValueError("run seed disagrees with config seed")
        model = _config_model(config)
        if model["model_key"] != "mentalroberta" or model["revision"] != EXPECTED_MODEL_REVISION:
            raise ValueError(f"model identity mismatch for seed {run['seed']}: {model}")
        if model != reference_model:
            raise ValueError(f"model source or identity mismatch for seed {run['seed']}")
        if config.get("max_length") != 512:
            raise ValueError(f"max_length mismatch for seed {run['seed']}")
        if bool(config.get("test_accessed")):
            raise ValueError(f"test_accessed must be false for seed {run['seed']}")
        if config.get("label2id") != expected_label2id:
            raise ValueError(f"label mapping mismatch for seed {run['seed']}")
        normalized_id2label = {str(key): value for key, value in config.get("id2label", {}).items()}
        if normalized_id2label != expected_id2label:
            raise ValueError(f"inverse label mapping mismatch for seed {run['seed']}")
        if config.get("data_sha256", {}) != reference_hashes:
            raise ValueError("dataset hashes do not match across seeds")
        integrity = run.get("integrity", {})
        if integrity.get("sha256") != config.get("data_sha256"):
            raise ValueError(f"dataset integrity hashes disagree with config for seed {run['seed']}")
        if bool(integrity.get("test_accessed", False)):
            raise ValueError(f"dataset integrity records test_accessed=true for seed {run['seed']}")
        integrity_rows = integrity.get("rows", {})
        if int(config.get("validation_rows_used", -1)) != expected_rows:
            raise ValueError(f"validation row count in config mismatch for seed {run['seed']}")
        if int(integrity_rows.get("validation", -1)) != expected_rows:
            raise ValueError(f"validation row count in integrity file mismatch for seed {run['seed']}")
        if int(integrity_rows.get("train", -1)) != int(config.get("train_rows_used", -2)):
            raise ValueError(f"training row counts disagree for seed {run['seed']}")
        if expected_train_rows is not None and int(config.get("train_rows_used", -1)) != expected_train_rows:
            raise ValueError(f"training row count mismatch for seed {run['seed']}")
        training = {key: config.get(key) for key in TRAINING_CONFIG_KEYS}
        if training != reference_training:
            raise ValueError(f"training configuration mismatch for seed {run['seed']}")
        predictions = run["predictions"]
        validate_predictions(predictions, LABELS, expected_rows=expected_rows)
        rows = predictions[["original_row", "true_id", "true_label"]].reset_index(drop=True)
        if not rows.equals(reference_rows):
            raise ValueError(f"validation row identities or true labels mismatch for seed {run['seed']}")
    return {
        "run_count": int(len(runs)),
        "observed_seeds": sorted(observed_seeds),
        "model": reference_model,
        "data_sha256": dict(reference_hashes),
        "training_config": reference_training,
        "validation_rows": int(expected_rows),
        "train_rows": int(reference_config.get("train_rows_used", 0)),
        "test_accessed": False,
        "all_hard_validations_pass": True,
    }


def _json_scalar(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"not JSON serializable: {type(value).__name__}")


def build_phase3_freeze(runs, source_paths=None, optional_summary=None):
    """Reconstruct the frozen Phase 3 inventory and aggregate metrics."""
    if not runs:
        raise ValueError("at least one run is required")
    ordered = sorted(runs, key=lambda item: item["seed"])
    expected = [run["seed"] for run in ordered]
    collection = validate_run_collection(
        ordered,
        expected_seeds=expected,
        expected_rows=len(ordered[0]["predictions"]),
    )
    inventory_rows = []
    metric_rows = []
    for run in ordered:
        inventory_rows.append(
            {
                "seed": int(run["seed"]),
                "run_id": run["run_id"],
                "archive_name": run["archive_name"],
                "archive_bytes": int(run["archive_bytes"]),
                "archive_sha256": run["archive_sha256"],
                "validated_manifest_members": int(run["validated_manifest_members"]),
                "validation_rows": int(len(run["predictions"])),
                "test_accessed": bool(run["config"].get("test_accessed")),
            }
        )
        row = {"seed": int(run["seed"]), "run_id": run["run_id"]}
        for key, value in run["stored_metrics"].items():
            if isinstance(value, (int, float)):
                row[key] = float(value)
        metric_rows.append(row)
    run_inventory = pd.DataFrame(inventory_rows).sort_values("seed").reset_index(drop=True)
    per_seed_metrics = pd.DataFrame(metric_rows).sort_values("seed").reset_index(drop=True)
    aggregate_rows = []
    for metric in [*METRIC_KEYS, "loss"]:
        values = per_seed_metrics[metric].to_numpy(dtype=float)
        aggregate_rows.append(
            {
                "metric": metric,
                "seed_count": int(len(values)),
                "mean": float(np.mean(values)),
                "sample_sd": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
                "minimum": float(np.min(values)),
                "maximum": float(np.max(values)),
            }
        )
    aggregate_metrics = pd.DataFrame(aggregate_rows)

    source_hashes = {}
    for name, path in sorted((source_paths or {}).items()):
        candidate = Path(path)
        source_hashes[name] = {
            "path": str(candidate),
            "sha256": sha256_file(candidate) if candidate.is_file() else None,
            "present": candidate.is_file(),
        }
    optional_summary_check = {"supplied": optional_summary is not None, "matched": None}
    if optional_summary is not None:
        macro = aggregate_metrics.set_index("metric").loc["macro_f1"]
        checks = []
        mappings = {
            "validation_macro_f1_mean": float(macro["mean"]),
            "validation_macro_f1_sample_sd": float(macro["sample_sd"]),
            "validation_accuracy_mean": float(
                aggregate_metrics.set_index("metric").loc["accuracy", "mean"]
            ),
            "completed_seeds": len(ordered),
            "seeds": [int(run["seed"]) for run in ordered],
            "model_key": collection["model"]["model_key"],
            "model_source": collection["model"]["source"],
            "model_revision": collection["model"]["revision"],
            "data_sha256": collection["data_sha256"],
            "test_accessed": False,
        }
        mappings.update(collection["training_config"])
        for key, expected_value in mappings.items():
            if key in optional_summary:
                observed = optional_summary[key]
                if isinstance(expected_value, float):
                    checks.append(bool(np.isclose(observed, expected_value, rtol=0.0, atol=1e-12)))
                else:
                    checks.append(observed == expected_value)
        optional_summary_check["matched"] = bool(checks and all(checks))
        optional_summary_check["checked_fields"] = sorted(
            key for key in mappings if key in optional_summary
        )
        if not optional_summary_check["matched"]:
            raise ValueError("optional Phase 3 summary does not match reconstructed results")

    reference = ordered[0]["config"]
    manifest = {
        "phase": "Phase 3 — Ten-Seed Principal Global Model",
        "status": "PASS",
        "created_utc": "2026-08-22T00:00:00Z",
        "creation_time_policy": "Fixed protocol-date timestamp for deterministic packaging",
        "expected_seeds": list(collection["observed_seeds"]),
        "observed_seeds": list(collection["observed_seeds"]),
        "completed_seeds": int(len(ordered)),
        "model": collection["model"],
        "label_order": list(LABELS),
        "training_config": collection["training_config"],
        "data_sha256": collection["data_sha256"],
        "train_rows": int(reference.get("train_rows_used", 0)),
        "validation_rows": int(reference.get("validation_rows_used", len(ordered[0]["predictions"]))),
        "input_archives": inventory_rows,
        "aggregate_metrics": {
            row["metric"]: {
                "mean": row["mean"],
                "sample_sd": row["sample_sd"],
                "minimum": row["minimum"],
                "maximum": row["maximum"],
            }
            for row in aggregate_rows
        },
        "optional_summary_check": optional_summary_check,
        "source_files": source_hashes,
        "model_archives_local": False,
        "model_archive_policy": "Model archives were intentionally not included in this local freeze.",
        "test_accessed": False,
        "all_hard_validations_pass": True,
    }
    return {
        "run_inventory": run_inventory,
        "per_seed_metrics": per_seed_metrics,
        "aggregate_metrics": aggregate_metrics,
        "manifest": manifest,
        "warnings": sorted(set(warning for run in ordered for warning in run.get("warnings", []))),
    }


def build_phase4_analysis(runs):
    """Build all Phase 4 per-seed and cross-seed boundary statistics."""
    ordered = sorted(runs, key=lambda item: item["seed"])
    pair_frames = []
    confusion_rows = []
    for run in ordered:
        seed = int(run["seed"])
        confusion = recompute_metrics(run["predictions"], LABELS)["confusion_matrix"]
        for true_index, true_label in enumerate(LABELS):
            for predicted_index, predicted_label in enumerate(LABELS):
                confusion_rows.append(
                    {
                        "seed": seed,
                        "true_label": true_label,
                        "predicted_label": predicted_label,
                        "count": int(confusion[true_index, predicted_index]),
                    }
                )
        pair_frames.append(rank_pairs(compute_pair_metrics(confusion, LABELS, seed)))
    per_seed_pairs = pd.concat(pair_frames, ignore_index=True).sort_values(
        ["seed", "rank", "pair"], kind="mergesort"
    ).reset_index(drop=True)
    stability = summarize_pairs(per_seed_pairs)
    rank_matrix = per_seed_pairs.pivot(index="pair", columns="seed", values="rank")
    rank_matrix = rank_matrix.reindex(sorted(rank_matrix.index)).reindex(
        columns=[run["seed"] for run in ordered]
    )
    rank_matrix.columns = [str(column) for column in rank_matrix.columns]
    agreement = kendalls_w(rank_matrix)
    spearman_rows, spearman_summary = pairwise_seed_spearman(rank_matrix)
    agreement["pairwise_spearman"] = spearman_summary

    candidates = stability[stability["top3_count"] >= 7].copy()
    candidates = candidates.sort_values(
        ["median_rank", "mean_bcr", "pair"], ascending=[True, False, True], kind="mergesort"
    )
    criterion1 = {
        "criterion": "top-three rank in at least 7 of 10 principal seeds",
        "threshold": 7,
        "candidate_pairs": candidates["pair"].tolist(),
        "candidate_count": int(len(candidates)),
        "final_hard_pairs_frozen": False,
        "phase5_required": True,
        "phase5_bootstrap_top3_frequency_threshold": 0.70,
        "interpretation": (
            "These pairs satisfy only criterion 1 and remain provisional until the "
            "predeclared Phase 5 bootstrap criterion is evaluated."
        ),
    }
    return {
        "per_seed_confusion_matrices": pd.DataFrame(confusion_rows),
        "per_seed_pair_metrics": per_seed_pairs,
        "pair_stability_summary": stability,
        "seed_rank_matrix": rank_matrix,
        "pairwise_seed_spearman": spearman_rows,
        "rank_agreement": agreement,
        "criterion1_candidates": criterion1,
    }


def _write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=_json_scalar) + "\n",
        encoding="utf-8",
    )


def _write_dataframe(path, frame):
    frame.to_csv(path, index=False, float_format="%.17g", lineterminator="\n")


def _write_figures(phase4, figures_dir):
    import matplotlib.pyplot as plt

    figures_dir.mkdir(parents=True, exist_ok=True)
    summary = phase4["pair_stability_summary"].sort_values("mean_bcr")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(summary["pair"], summary["mean_bcr"], xerr=summary["sample_sd_bcr"], capsize=3)
    ax.set_xlabel("Mean bidirectional confusion rate")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 mean BCR across ten principal seeds")
    fig.tight_layout()
    fig.savefig(figures_dir / "mean_bcr_by_pair.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    error_summary = phase4["pair_stability_summary"].sort_values("mean_pair_error_share")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(error_summary["pair"], error_summary["mean_pair_error_share"])
    ax.set_xlabel("Mean share of all validation errors")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 error concentration by pair")
    fig.tight_layout()
    fig.savefig(figures_dir / "error_share_by_pair.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    rank_matrix = phase4["seed_rank_matrix"]
    fig, ax = plt.subplots(figsize=(10, 7))
    image = ax.imshow(rank_matrix.to_numpy(dtype=float), aspect="auto", cmap="viridis_r", vmin=1, vmax=10)
    ax.set_xticks(range(len(rank_matrix.columns)), rank_matrix.columns)
    ax.set_yticks(range(len(rank_matrix.index)), rank_matrix.index)
    ax.set_xlabel("Seed")
    ax.set_ylabel("Class pair")
    ax.set_title("Phase 4 BCR ranks (lower is harder)")
    fig.colorbar(image, ax=ax, label="Rank")
    fig.tight_layout()
    fig.savefig(figures_dir / "pair_rank_heatmap.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def write_sha256_manifest(root, output_path=None):
    root = Path(root)
    output_path = Path(output_path) if output_path else root / "sha256_manifest.csv"
    rows = []
    for path in sorted(candidate for candidate in root.rglob("*") if candidate.is_file() and candidate != output_path):
        rows.append(
            {
                "path": path.relative_to(root).as_posix(),
                "sha256": sha256_file(path),
                "bytes": int(path.stat().st_size),
            }
        )
    pd.DataFrame(rows, columns=["path", "sha256", "bytes"]).to_csv(
        output_path, index=False, lineterminator="\n"
    )
    return rows


def write_results(phase3, phase4, output_dir, create_figures=True):
    """Write the approved Phase 3 freeze and Phase 4 artifact tree."""
    output_dir = Path(output_dir)
    phase3_dir = output_dir / "phase3_freeze"
    phase4_dir = output_dir / "phase4"
    figures_dir = phase4_dir / "figures"
    phase3_dir.mkdir(parents=True, exist_ok=True)
    phase4_dir.mkdir(parents=True, exist_ok=True)

    _write_dataframe(phase3_dir / "phase3_run_inventory.csv", phase3["run_inventory"])
    _write_dataframe(phase3_dir / "phase3_per_seed_metrics.csv", phase3["per_seed_metrics"])
    _write_dataframe(phase3_dir / "phase3_aggregate_metrics.csv", phase3["aggregate_metrics"])
    _write_json(phase3_dir / "phase3_freeze_manifest.json", phase3["manifest"])
    macro = phase3["aggregate_metrics"].set_index("metric").loc["macro_f1"]
    accuracy = phase3["aggregate_metrics"].set_index("metric").loc["accuracy"]
    phase3_report = (
        "# Phase 3 Freeze Report\n\n"
        f"Status: **{phase3['manifest']['status']}**\n\n"
        f"Completed seeds: {phase3['manifest']['completed_seeds']}\n\n"
        f"Validation Macro-F1: {macro['mean']:.9f} ± {macro['sample_sd']:.9f} (sample SD)\n\n"
        f"Validation accuracy: {accuracy['mean']:.9f} ± {accuracy['sample_sd']:.9f} (sample SD)\n\n"
        "The locked test was not accessed. Model archives were intentionally not included locally.\n"
    )
    (phase3_dir / "phase3_freeze_report.md").write_text(phase3_report, encoding="utf-8")

    _write_dataframe(
        phase4_dir / "phase4_per_seed_confusion_matrices.csv",
        phase4["per_seed_confusion_matrices"],
    )
    _write_dataframe(
        phase4_dir / "phase4_per_seed_pair_metrics.csv",
        phase4["per_seed_pair_metrics"],
    )
    _write_dataframe(
        phase4_dir / "phase4_pair_stability_summary.csv",
        phase4["pair_stability_summary"],
    )
    rank_export = phase4["seed_rank_matrix"].reset_index()
    _write_dataframe(phase4_dir / "phase4_seed_rank_matrix.csv", rank_export)
    _write_dataframe(
        phase4_dir / "phase4_pairwise_seed_spearman.csv",
        phase4["pairwise_seed_spearman"],
    )
    _write_json(phase4_dir / "phase4_rank_agreement.json", phase4["rank_agreement"])
    _write_json(
        phase4_dir / "phase4_criterion1_candidates.json",
        phase4["criterion1_candidates"],
    )
    top_rows = phase4["pair_stability_summary"].head(3)
    top_lines = "\n".join(
        f"- {row.pair}: mean BCR {row.mean_bcr:.6f}, top-three count {int(row.top3_count)}/10"
        for row in top_rows.itertuples()
    )
    phase4_report = (
        "# Phase 4 Residual Boundary Report\n\n"
        "This report identifies **provisional criterion-1 candidates only**. "
        "No final hard pair is frozen here; Phase 5 bootstrap validation is mandatory.\n\n"
        "## Leading observed pairs\n\n"
        f"{top_lines}\n\n"
        f"Kendall's W: {phase4['rank_agreement']['kendalls_w']:.6f}\n"
    )
    (phase4_dir / "phase4_report.md").write_text(phase4_report, encoding="utf-8")
    if create_figures:
        _write_figures(phase4, figures_dir)

    execution_summary = {
        "status": "PASS",
        "archive_count": int(len(phase3["run_inventory"])),
        "completed_seeds": int(phase3["manifest"]["completed_seeds"]),
        "all_hard_validations_pass": True,
        "test_accessed": False,
        "validation_macro_f1_mean": float(macro["mean"]),
        "validation_macro_f1_sample_sd": float(macro["sample_sd"]),
        "validation_accuracy_mean": float(accuracy["mean"]),
        "kendalls_w": float(phase4["rank_agreement"]["kendalls_w"]),
        "criterion1_candidates": phase4["criterion1_candidates"]["candidate_pairs"],
        "final_hard_pairs_frozen": False,
        "phase5_required": True,
        "warnings": list(phase3.get("warnings", [])),
    }
    _write_json(output_dir / "phase3_phase4_execution_summary.json", execution_summary)
    manifest_rows = write_sha256_manifest(output_dir)
    return {
        "output_dir": output_dir,
        "manifest_entries": int(len(manifest_rows)),
        "execution_summary": execution_summary,
    }


def write_deterministic_zip(source_dir, zip_path, prefix="phase4_boundary_analysis_results"):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    fixed_time = (2020, 1, 1, 0, 0, 0)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(candidate for candidate in source_dir.rglob("*") if candidate.is_file()):
            member = (Path(prefix) / path.relative_to(source_dir)).as_posix()
            info = zipfile.ZipInfo(member, fixed_time)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, path.read_bytes())
    return zip_path


In [ ]:
# Embedded deterministic self-checks.
_toy = compute_pair_metrics(np.array([[8, 2], [3, 7]]), ["A", "B"], seed=13).iloc[0]
assert abs(_toy["bcr"] - 0.25) <= 1e-15
_identical_ranks = pd.DataFrame({seed: np.arange(1, 11, dtype=float) for seed in range(10)})
assert abs(kendalls_w(_identical_ranks)["kendalls_w"] - 1.0) <= 1e-15
print("Embedded statistical self-checks: PASS")


Embedded statistical self-checks: PASS


# 5. Input discovery and archive-integrity validation


In [ ]:
archives = discover_archives(INPUT_DIR)
archive_checks = pd.DataFrame([validate_archive(path) for path in archives])
archive_checks["archive_path"] = archive_checks["archive_path"].astype(str)
assert len(archive_checks) == 10
display(archive_checks[["archive_name", "archive_bytes", "archive_sha256", "validated_manifest_members"]])
print("Archive integrity gate: PASS")


,archive_name,archive_bytes,archive_sha256,validated_manifest_members
0,mentalroberta_full_seed1024_len512_e91f034d8a7...,439480,4ebc235f48f1b990bfff860f758729510de544b724da83...,13
1,mentalroberta_full_seed123_len512_d1ad313ee75d...,437881,283b063df338d1ce50ca9dc60d49f027e91ae7cad024fe...,13
2,mentalroberta_full_seed13_len512_fea2991778f3_...,441549,23ceddd17dfe24eff0f1367824f8a020d83e606da643b2...,13
3,mentalroberta_full_seed2026_len512_f6630d94239...,435153,4d07884013a0f5af78f724e33e95a4ebe681a7856e4059...,13
4,mentalroberta_full_seed211_len512_149308113b40...,438534,56a312411d83f0325d2f6ad107f2c873c69133cd426b4f...,13
5,mentalroberta_full_seed3407_len512_d8596ea710c...,436962,e8d96433959e8113700e980a8d31c03bd4ee1ea16ddb95...,13
6,mentalroberta_full_seed340_len512_6353292c8230...,429349,fff4222a9a65067fcd8e3a8c0944e63437d9acd55e353c...,13
7,mentalroberta_full_seed42_len512_be72f916ca15_...,444578,c45644bf88c9c813742de5613d7bd53c04c65d48f1b302...,13
8,mentalroberta_full_seed512_len512_412e1caa7392...,439735,9db1ec1a67952c3868160ed365b03052839ac936c1d996...,13
9,mentalroberta_full_seed77_len512_d14375990239_...,440261,6e3d4ae16d9cff4c4a94ba3964b4c16c434dbb4b4a3c90...,13


Archive integrity gate: PASS


# 6. Prediction-level consistency validation


In [ ]:
runs = [load_run(path, expected_rows=1490) for path in archives]
collection_validation = validate_run_collection(
    runs,
    expected_seeds=EXPECTED_SEEDS,
    expected_rows=1490,
    expected_train_rows=6954,
)
assert collection_validation["all_hard_validations_pass"] is True
assert collection_validation["test_accessed"] is False
print(json.dumps(collection_validation, indent=2, sort_keys=True))


{
  "all_hard_validations_pass": true,
  "data_sha256": {
    "train": "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b",
    "validation": "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
  },
  "model": {
    "model_key": "mentalroberta",
    "revision": "610e9d35d8f2550d713579077242d5e523209cea",
    "source": "mental/mental-roberta-base"
  },
  "observed_seeds": [
    13,
    42,
    77,
    123,
    211,
    340,
    512,
    1024,
    2026,
    3407
  ],
  "run_count": 10,
  "test_accessed": false,
  "train_rows": 6954,
  "training_config": {
    "epochs": 8,
    "eval_batch_size": 16,
    "grad_accum_steps": 4,
    "learning_rate": 2e-05,
    "max_grad_norm": 1.0,
    "max_length": 512,
    "patience": 3,
    "run_mode": "full",
    "train_batch_size": 8,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01
  },
  "validation_rows": 1490
}


# 7. Phase 3 freeze reconstruction


In [ ]:
optional_summary = None
if OPTIONAL_SUMMARY_PATH.is_file():
    optional_summary = json.loads(OPTIONAL_SUMMARY_PATH.read_text(encoding="utf-8"))

source_paths = {
    "phase3_training_notebook": SOURCE_NOTEBOOK_PATH,
    "master_research_plan": MASTER_PLAN_PATH,
}
phase3 = build_phase3_freeze(
    runs,
    source_paths=source_paths,
    optional_summary=optional_summary,
)
assert phase3["manifest"]["status"] == "PASS"
assert phase3["manifest"]["test_accessed"] is False
display(phase3["per_seed_metrics"])
display(phase3["aggregate_metrics"])


,seed,run_id,accuracy,loss,macro_f1,macro_precision,macro_recall,mcc,weighted_f1
0,13,mentalroberta_full_seed13_len512_fea2991778f3,0.872483,0.419653,0.869344,0.874291,0.865919,0.839270,0.872029
1,42,mentalroberta_full_seed42_len512_be72f916ca15,0.866443,0.438105,0.863804,0.864104,0.863672,0.831750,0.866546
2,77,mentalroberta_full_seed77_len512_d14375990239,0.869128,0.573292,0.866476,0.866514,0.868129,0.835794,0.868453
3,123,mentalroberta_full_seed123_len512_d1ad313ee75d,0.862416,0.749883,0.859565,0.859903,0.860073,0.826905,0.861821
4,211,mentalroberta_full_seed211_len512_149308113b40,0.870470,0.511067,0.865441,0.869623,0.863558,0.836960,0.869169
5,340,mentalroberta_full_seed340_len512_6353292c8230,0.864430,0.429781,0.862736,0.866327,0.859982,0.829147,0.864698
6,512,mentalroberta_full_seed512_len512_412e1caa7392,0.866443,0.651294,0.863301,0.862724,0.864174,0.831904,0.866136
7,1024,mentalroberta_full_seed1024_len512_e91f034d8a71,0.867114,0.603166,0.863884,0.868793,0.861606,0.832917,0.866240
8,2026,mentalroberta_full_seed2026_len512_f6630d942394,0.863758,0.774403,0.860511,0.861201,0.860565,0.828542,0.863219
9,3407,mentalroberta_full_seed3407_len512_d8596ea710c3,0.863087,0.743469,0.856917,0.863193,0.854547,0.827735,0.861158


,metric,seed_count,mean,sample_sd,minimum,maximum
0,accuracy,10,0.866577,0.003315,0.862416,0.872483
1,macro_precision,10,0.865667,0.004363,0.859903,0.874291
2,macro_recall,10,0.862223,0.003770,0.854547,0.868129
3,macro_f1,10,0.863198,0.003567,0.856917,0.869344
4,weighted_f1,10,0.865947,0.003384,0.861158,0.872029
5,mcc,10,0.832092,0.004176,0.826905,0.839270
6,loss,10,0.589411,0.137968,0.419653,0.774403


# 8. Phase 3 freeze artifact export


In [ ]:
phase3_preview = {
    "status": phase3["manifest"]["status"],
    "completed_seeds": phase3["manifest"]["completed_seeds"],
    "model": phase3["manifest"]["model"],
    "data_sha256": phase3["manifest"]["data_sha256"],
    "test_accessed": phase3["manifest"]["test_accessed"],
}
print(json.dumps(phase3_preview, indent=2, sort_keys=True))


{
  "completed_seeds": 10,
  "data_sha256": {
    "train": "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b",
    "validation": "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"
  },
  "model": {
    "model_key": "mentalroberta",
    "revision": "610e9d35d8f2550d713579077242d5e523209cea",
    "source": "mental/mental-roberta-base"
  },
  "status": "PASS",
  "test_accessed": false
}


# 9. Phase 4 per-seed confusion and BCR calculation


In [ ]:
phase4 = build_phase4_analysis(runs)
assert len(phase4["per_seed_pair_metrics"]) == 100
assert len(phase4["per_seed_confusion_matrices"]) == 250
display(phase4["per_seed_pair_metrics"].head(20))


,seed,pair,class_i,class_j,support_i,support_j,confusion_i_to_j,confusion_j_to_i,pair_errors,bcr,total_errors,pair_error_share,rank
0,13,Frustrated|Others,Frustrated,Others,277,381,30,26,56,0.088272,190,0.294737,1.0
1,13,Suicidal|Depressed,Suicidal,Depressed,319,219,15,24,39,0.078305,190,0.205263,2.0
2,13,Depressed|Others,Depressed,Others,219,381,18,3,21,0.045033,190,0.110526,3.0
3,13,Anxious|Others,Anxious,Others,294,381,9,9,18,0.027117,190,0.094737,4.0
4,13,Suicidal|Frustrated,Suicidal,Frustrated,319,277,5,9,14,0.024082,190,0.073684,5.0
5,13,Suicidal|Others,Suicidal,Others,319,381,7,9,16,0.022783,190,0.084211,6.0
6,13,Depressed|Anxious,Depressed,Anxious,219,294,5,6,11,0.021620,190,0.057895,7.0
7,13,Depressed|Frustrated,Depressed,Frustrated,219,277,4,3,7,0.014548,190,0.036842,8.0
8,13,Suicidal|Anxious,Suicidal,Anxious,319,294,2,3,5,0.008237,190,0.026316,9.0
9,13,Anxious|Frustrated,Anxious,Frustrated,294,277,2,1,3,0.005206,190,0.015789,10.0


# 10. Cross-seed stability summaries


In [ ]:
display(phase4["pair_stability_summary"])
display(phase4["seed_rank_matrix"])


,pair,seed_count,mean_bcr,sample_sd_bcr,median_bcr,bcr_q1,bcr_q3,mean_rank,median_rank,top1_count,top1_frequency,top3_count,top3_frequency,mean_pair_errors,mean_pair_error_share
0,Frustrated|Others,10,0.093227,0.004661,0.091880,0.089996,0.094875,1.0,1.0,10,1.0,10,1.0,59.7,0.300273
1,Suicidal|Depressed,10,0.078162,0.004147,0.077880,0.076534,0.080163,2.0,2.0,0,0.0,10,1.0,39.0,0.196256
2,Depressed|Others,10,0.046388,0.006627,0.045545,0.042889,0.051977,3.0,3.0,0,0.0,10,1.0,24.4,0.122482
3,Depressed|Anxious,10,0.026351,0.004783,0.025312,0.023903,0.029442,4.7,4.0,0,0.0,0,0.0,13.2,0.066373
4,Anxious|Others,10,0.025645,0.004446,0.024640,0.022648,0.027020,4.7,5.0,0,0.0,0,0.0,17.5,0.087997
5,Suicidal|Others,10,0.020884,0.001763,0.020833,0.019721,0.022391,6.4,6.0,0,0.0,0,0.0,15.0,0.075519
6,Suicidal|Frustrated,10,0.021361,0.003776,0.020472,0.018026,0.023750,6.5,6.5,0,0.0,0,0.0,12.4,0.062395
7,Depressed|Frustrated,10,0.015763,0.003207,0.014548,0.013711,0.018065,7.8,8.0,0,0.0,0,0.0,7.7,0.038794
8,Suicidal|Anxious,10,0.010341,0.003239,0.009021,0.008237,0.011538,9.0,9.0,0,0.0,0,0.0,6.3,0.031778
9,Anxious|Frustrated,10,0.006321,0.001233,0.006161,0.005311,0.007011,9.9,10.0,0,0.0,0,0.0,3.6,0.018133


,13,42,77,123,211,340,512,1024,2026,3407
pair,,,,,,,,,,
Anxious|Frustrated,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,9.0,10.0
Anxious|Others,4.0,4.0,5.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0
Depressed|Anxious,7.0,5.0,4.0,4.0,4.0,7.0,4.0,4.0,4.0,4.0
Depressed|Frustrated,8.0,8.0,8.0,8.0,7.0,8.0,8.0,9.0,8.0,6.0
Depressed|Others,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
Frustrated|Others,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Suicidal|Anxious,9.0,9.0,9.0,9.0,9.0,9.0,9.0,8.0,10.0,9.0
Suicidal|Depressed,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
Suicidal|Frustrated,5.0,7.0,6.0,7.0,8.0,5.0,6.0,7.0,6.0,8.0


# 11. Kendall concordance and pairwise Spearman analysis


In [ ]:
print(json.dumps(phase4["rank_agreement"], indent=2, sort_keys=True))
display(phase4["pairwise_seed_spearman"])
assert len(phase4["pairwise_seed_spearman"]) == 45


{
  "kendalls_w": 0.9544242424242424,
  "method": "Kendall coefficient of concordance with tie correction",
  "pair_count": 10,
  "pairwise_spearman": {
    "comparison_count": 45,
    "maximum_spearman_rho": 0.9999999999999999,
    "median_spearman_rho": 0.9515151515151514,
    "method": "Pearson correlation applied to complete average-rank vectors",
    "minimum_spearman_rho": 0.8545454545454544,
    "spearman_rho_q1": 0.9272727272727272,
    "spearman_rho_q3": 0.9757575757575757
  },
  "seed_count": 10,
  "tie_correction": 0.0
}


,seed_a,seed_b,spearman_rho
0,13,42,0.951515
1,13,77,0.927273
2,13,123,0.915152
3,13,211,0.878788
4,13,340,1.000000
5,13,512,0.927273
6,13,1024,0.903030
7,13,2026,0.915152
8,13,3407,0.854545
9,42,77,0.975758


# 12. Tables and figures


In [ ]:
write_state = write_results(phase3, phase4, OUTPUT_DIR, create_figures=True)
display(phase4["pair_stability_summary"].head(10))
print(f"Generated {write_state['manifest_entries']} manifest-covered output files.")


,pair,seed_count,mean_bcr,sample_sd_bcr,median_bcr,bcr_q1,bcr_q3,mean_rank,median_rank,top1_count,top1_frequency,top3_count,top3_frequency,mean_pair_errors,mean_pair_error_share
0,Frustrated|Others,10,0.093227,0.004661,0.091880,0.089996,0.094875,1.0,1.0,10,1.0,10,1.0,59.7,0.300273
1,Suicidal|Depressed,10,0.078162,0.004147,0.077880,0.076534,0.080163,2.0,2.0,0,0.0,10,1.0,39.0,0.196256
2,Depressed|Others,10,0.046388,0.006627,0.045545,0.042889,0.051977,3.0,3.0,0,0.0,10,1.0,24.4,0.122482
3,Depressed|Anxious,10,0.026351,0.004783,0.025312,0.023903,0.029442,4.7,4.0,0,0.0,0,0.0,13.2,0.066373
4,Anxious|Others,10,0.025645,0.004446,0.024640,0.022648,0.027020,4.7,5.0,0,0.0,0,0.0,17.5,0.087997
5,Suicidal|Others,10,0.020884,0.001763,0.020833,0.019721,0.022391,6.4,6.0,0,0.0,0,0.0,15.0,0.075519
6,Suicidal|Frustrated,10,0.021361,0.003776,0.020472,0.018026,0.023750,6.5,6.5,0,0.0,0,0.0,12.4,0.062395
7,Depressed|Frustrated,10,0.015763,0.003207,0.014548,0.013711,0.018065,7.8,8.0,0,0.0,0,0.0,7.7,0.038794
8,Suicidal|Anxious,10,0.010341,0.003239,0.009021,0.008237,0.011538,9.0,9.0,0,0.0,0,0.0,6.3,0.031778
9,Anxious|Frustrated,10,0.006321,0.001233,0.006161,0.005311,0.007011,9.9,10.0,0,0.0,0,0.0,3.6,0.018133


Generated 17 manifest-covered output files.


# 13. Phase 4 status and criterion-1 candidate report


In [ ]:
criterion1 = phase4["criterion1_candidates"]
assert criterion1["final_hard_pairs_frozen"] is False
assert criterion1["phase5_required"] is True
print(json.dumps(criterion1, indent=2, sort_keys=True))


{
  "candidate_count": 3,
  "candidate_pairs": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "criterion": "top-three rank in at least 7 of 10 principal seeds",
  "final_hard_pairs_frozen": false,
  "interpretation": "These pairs satisfy only criterion 1 and remain provisional until the predeclared Phase 5 bootstrap criterion is evaluated.",
  "phase5_bootstrap_top3_frequency_threshold": 0.7,
  "phase5_required": true,
  "threshold": 7
}


# 14. Output manifest, deterministic archive, and final execution summary


In [ ]:
manifest = pd.read_csv(OUTPUT_DIR / "sha256_manifest.csv")
for row in manifest.itertuples(index=False):
    path = OUTPUT_DIR / row.path
    assert path.is_file(), path
    assert path.stat().st_size == row.bytes, path
    assert sha256_file(path) == row.sha256, path

write_deterministic_zip(OUTPUT_DIR, RESULT_ZIP_PATH)
with tempfile.TemporaryDirectory() as temporary_directory:
    comparison_zip = Path(temporary_directory) / "comparison.zip"
    write_deterministic_zip(OUTPUT_DIR, comparison_zip)
    assert sha256_file(RESULT_ZIP_PATH) == sha256_file(comparison_zip)

final_summary = write_state["execution_summary"]
assert final_summary["archive_count"] == 10
assert final_summary["completed_seeds"] == 10
assert final_summary["all_hard_validations_pass"] is True
assert final_summary["test_accessed"] is False
assert final_summary["phase5_required"] is True
assert final_summary["final_hard_pairs_frozen"] is False

print(json.dumps({
    **final_summary,
    "result_zip": str(RESULT_ZIP_PATH),
    "result_zip_sha256": sha256_file(RESULT_ZIP_PATH),
    "result_zip_bytes": RESULT_ZIP_PATH.stat().st_size,
}, indent=2, sort_keys=True))
print("PHASE 3 FREEZE + PHASE 4 CPU ANALYSIS: PASS")


{
  "all_hard_validations_pass": true,
  "archive_count": 10,
  "completed_seeds": 10,
  "criterion1_candidates": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "final_hard_pairs_frozen": false,
  "kendalls_w": 0.9544242424242424,
  "phase5_required": true,
  "result_zip": "mentalroberta_phase3_freeze_phase4_boundary_analysis_RESULTS.zip",
  "result_zip_bytes": 248325,
  "result_zip_sha256": "ad1611cb441efbc33f7d9a5f1e35975aa0782f867b4be6edcd9104b2673542c3",
  "status": "PASS",
  "test_accessed": false,
  "validation_accuracy_mean": 0.8665771812080537,
  "validation_macro_f1_mean": 0.8631978431731311,
  "validation_macro_f1_sample_sd": 0.003566556282862149,
  "warnings": [
    "full-run repeated-fit reproducibility was not performed"
  ]
}
PHASE 3 FREEZE + PHASE 4 CPU ANALYSIS: PASS
